# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. `20 10`.
По ярлыку `"b"` получается 20, а `iloc[0]` берет значение на первой позиции 10

2. `Алматы NaN`, `Астана NaN`, `Шымкент 3.0`.
Pandas складывает Series по ярлыкам, то есть, если города нет во второй Series, то результат будет `NaN`

3. `(2, 2)` - `x > 1` оставляет две строки, а столбцов остается два

4. `Series DataFrame`. Один столбец через `df["x"]` — это Series, список из одного столбца через `df[["x"]]` — DataFrame

5. `[1, 2] 5`. После фильтра индексы сохраняются, то есть, у первой оставшейся строки (под индексом 1) второй столбец `y` равен 5

In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [4]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [8]:
print("Строк и столбцов:", df.shape)
print("Типы столбцов:")
print(df.dtypes)
print("Пропусков в delivery_days: ", df["delivery_days"].isna().sum())
print("Разных значений в city: ", df["city"].nunique())
print("Разных значений category: ", df["category"].nunique())
print("Разных значений status: ", df["status"].nunique())
print("Разных значений customer_id: ", df["customer_id"].nunique())

print("Повторов ID заказов: ", len(df) - df["order_id"].nunique())
print("Тестовых заказов: ", (df["customer_id"] == "TEST").sum())

Строк и столбцов: (5463, 8)
Типы столбцов:
order_id           int64
order_date        object
customer_id       object
city              object
category          object
amount           float64
delivery_days    float64
status            object
dtype: object
Пропусков в delivery_days:  160
Разных значений в city:  14
Разных значений category:  5
Разных значений status:  2
Разных значений customer_id:  2003
Повторов ID заказов:  180
Тестовых заказов:  3


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [9]:
orders_paid = df.loc[paid & not_test, "amount"]
print("Оплаченных заказов без TEST: ", len(orders_paid))
print("Средний чек: ", round(orders_paid.mean(), 2))
print("Медиана: ", orders_paid.median())
print("25-й перцентиль: ", orders_paid.quantile(0.25))
print("75-й перцентиль: ", orders_paid.quantile(0.75))

Оплаченных заказов без TEST:  4901
Средний чек:  14672.39
Медиана:  10300.0
25-й перцентиль:  5880.0
75-й перцентиль:  18690.0


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [13]:
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]
sh_mask = df["city"].isin(SH)
al_mask = df["city"].isin(AL)
cancelled = df["status"] == "отменён"
all_rate = cancelled.mean() * 100
sh_rate = (cancelled & sh_mask).sum() / sh_mask.sum() * 100
al_rate = (cancelled & al_mask).sum() / al_mask.sum() * 100
print("Доля отмененных заказов во всех городах в %: ", round(all_rate, 2))
print("Доля отмененных заказов в Шымкенте в %: ", round(sh_rate, 2))
print("Доля отмененных заказов в Алматы в %: ", round(al_rate, 2))
print("Во сколько раз доля в Шымкенте выше: ", round(sh_rate / al_rate, 2))

Доля отмененных заказов во всех городах в %:  10.23
Доля отмененных заказов в Шымкенте в %:  20.4
Доля отмененных заказов в Алматы в %:  7.86
Во сколько раз доля в Шымкенте выше:  2.59


Доля в Шымкенте выше в 2.59 раза



## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [15]:
for city_name, city_mask in [("Шымкент", sh_mask), ("Алматы", al_mask)]:
    sep_revenue = df.loc[paid & not_test & sep & city_mask, "amount"].sum()
    oct_revenue = df.loc[paid & not_test & octo & city_mask, "amount"].sum()
    change = (oct_revenue - sep_revenue) / sep_revenue * 100
    print(city_name, "—", "сентябрь:", sep_revenue, ";", "октябрь:", oct_revenue,
          ";", "изменение в %: ", round(change, 2))

Шымкент — сентябрь: 4949800.0 ; октябрь: 4044010.0 ; изменение в %:  -18.3
Алматы — сентябрь: 15523940.0 ; октябрь: 14643020.0 ; изменение в %:  -5.67


С учетом тестовых данных в Шымкенте падение продаж в сентябре больше, чем в Алматы

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [16]:
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print("Отмененных заказов в Шымкенте: ", len(cancel_sh))

Отмененных заказов в Шымкенте:  164


Какие были ошибки:
1.   были упущены скобки при мерджинге условий
2.   не учитывалось разнре написание наименования города



## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [18]:
client_counts = df["customer_id"].value_counts()
top_client = client_counts.index[0]
client_orders = df.loc[df["customer_id"] == top_client,
                       ["order_id", "order_date", "city", "amount", "status"]]
print("Больше всего заказов у клиента: ", top_client, "—", client_counts.iloc[0], "заказов")
print(client_orders)
print("Число повторяющихся order_id у топ клиента:", len(client_orders) - client_orders["order_id"].nunique())

Больше всего заказов у клиента:  C01818 — 11 заказов
      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен
Число повторяющихся order_id у топ клиента: 1


Есои есть повторы order_id, значит есть дубли, которые надо очищать

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [5]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [6]:
# 8